<a href="https://colab.research.google.com/github/rodrigorissettoterra/buscador_adaptativo_contextual/blob/main/buscador_adaptativo_contextual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Buscador Adaptativo Contextual — v5

O usuário digita **qualquer assunto**. O sistema interpreta o contexto, descobre fontes na web, classifica o tipo de cada fonte e seleciona dinamicamente **5 sites relevantes e complementares**.

### Fluxo

```text
TERMO
  ↓
CONTEXTO / INTENÇÃO
  ↓
BUSCA AMPLA + CONSULTAS AUXILIARES
  ↓
TIPO DE FONTE + RELEVÂNCIA SEMÂNTICA
  ↓
SELEÇÃO DIVERSIFICADA DOS 5 SITES
  ↓
BUSCA DENTRO DE CADA SITE
  ↓
EXTRAÇÃO + RANKING + DEDUPLICAÇÃO
  ↓
SÍNTESE EXTRATIVA + REFERÊNCIAS
```

### Três modos

- **Relevância máxima**: interfere pouco no ranking original.
- **Equilibrada**: padrão recomendado; equilibra relevância e diversidade.
- **Descoberta / web independente**: aumenta moderadamente a chance de fontes especializadas, blogs e nichos entrarem quando forem relevantes.

A classificação de fonte é uma **estimativa heurística**, usada para diversificação — não uma afirmação absoluta sobre autoridade ou qualidade.

In [1]:
!pip -q install -U ddgs sentence-transformers trafilatura tldextract pandas openpyxl ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.

## 1. Imports, modelo e configurações

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from urllib.robotparser import RobotFileParser
from collections import defaultdict
import hashlib
import html
import math
import re
import time

import numpy as np
import pandas as pd
import requests
import tldextract
import trafilatura

from ddgs import DDGS
from sentence_transformers import SentenceTransformer
from IPython.display import display, clear_output, Markdown, HTML
import ipywidgets as widgets

USER_AGENT = "Mozilla/5.0 (compatible; AdaptiveContextSearch/1.0)"
REGION = "br-pt"
TIMEOUT = 15
TOP_SITES = 5
MAX_EXTRACTED_CHARS = 7000

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
})

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
semantic_model = SentenceTransformer(MODEL_NAME)

print("Modelo carregado:", MODEL_NAME)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 2. Modelos de dados e utilitários

In [3]:
@dataclass
class WebResult:
    rank: int
    title: str
    url: str
    snippet: str
    domain: str
    discovery_query: str
    semantic_score: float = 0.0


@dataclass
class SourceProfile:
    source_type: str
    confidence: float
    niche_signal: float
    evidence: list[str] = field(default_factory=list)


@dataclass
class SiteCandidate:
    domain: str
    source_type: str
    type_confidence: float
    niche_signal: float
    base_score: float
    contextual_score: float
    selection_score: float
    best_rank: int
    appearances: int
    query_coverage: int
    best_semantic_score: float
    average_semantic_score: float
    example_title: str
    example_url: str
    why_selected: str = ""
    descriptor: str = ""


@dataclass
class FinalResult:
    source_id: str
    site: str
    source_type: str
    title: str
    url: str
    snippet: str
    extracted_text: str
    extraction_status: str
    site_score: float
    semantic_score: float
    original_rank: int


TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign",
    "utm_term", "utm_content", "gclid", "fbclid",
}


def clean_text(value: Optional[str]) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def registered_domain(url: str) -> str:
    ext = tldextract.extract(url)
    if ext.domain and ext.suffix:
        return f"{ext.domain}.{ext.suffix}".lower()
    return urlsplit(url).netloc.lower()


def canonical_url(url: str) -> str:
    try:
        parts = urlsplit(url)
        query = [
            (k, v)
            for k, v in parse_qsl(parts.query, keep_blank_values=True)
            if k.lower() not in TRACKING_PARAMS
        ]
        return urlunsplit((
            parts.scheme.lower(),
            parts.netloc.lower(),
            parts.path.rstrip("/") or "/",
            urlencode(query),
            "",
        ))
    except Exception:
        return url


def short_hash(text: str) -> str:
    return hashlib.sha256(
        text.encode("utf-8", errors="ignore")
    ).hexdigest()[:8]


def embed_texts(texts: list[str]) -> np.ndarray:
    if not texts:
        return np.empty((0, 0))
    return semantic_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )


def semantic_scores(query: str, documents: list[str]) -> list[float]:
    if not documents:
        return []
    q = embed_texts([query])[0]
    d = embed_texts(documents)
    return [
        round(max(0.0, min(1.0, float(v))) * 100, 2)
        for v in d @ q
    ]

## 3. Interpretação semântica da intenção

A intenção só orienta a descoberta. O usuário continua digitando apenas o assunto.

Categorias:
- informação geral;
- pesquisa científica;
- notícias;
- prática/tutorial;
- técnica/documentação;
- compra/comparação;
- opinião/comunidade.

In [4]:
INTENT_PROTOTYPES = {
    "Informação geral":
        "explicação conceitual, definição, visão geral, entender um assunto",
    "Pesquisa científica":
        "artigo científico, estudo, pesquisa acadêmica, evidência, revisão sistemática",
    "Notícias / atualidade":
        "notícias recentes, fatos atuais, atualização, cobertura jornalística",
    "Prática / tutorial":
        "como fazer, passo a passo, tutorial, guia prático, experiência prática",
    "Técnica / documentação":
        "documentação técnica, referência, especificação, manual, API, engenharia",
    "Compra / comparação":
        "comprar, preço, melhor produto, comparação, avaliação, custo benefício",
    "Opinião / comunidade":
        "opiniões, relatos, discussão, fórum, comunidade, experiência de usuários",
}

INTENT_EXPANSIONS = {
    "Informação geral": ["explicação análise", "guia referência"],
    "Pesquisa científica": ["estudo pesquisa evidências", "artigo revisão literatura"],
    "Notícias / atualidade": ["notícias análise", "atualização recente"],
    "Prática / tutorial": ["guia passo a passo", "experiência prática tutorial"],
    "Técnica / documentação": ["documentação referência técnica", "guia especificação"],
    "Compra / comparação": ["comparação avaliação", "preço custo benefício"],
    "Opinião / comunidade": ["opinião experiência", "discussão comunidade"],
}


def infer_search_intent(query: str) -> dict:
    labels = list(INTENT_PROTOTYPES)
    descriptions = [INTENT_PROTOTYPES[label] for label in labels]

    query_embedding = embed_texts([query])[0]
    intent_embeddings = embed_texts(descriptions)
    similarities = intent_embeddings @ query_embedding
    order = np.argsort(similarities)[::-1]

    best = int(order[0])
    second = int(order[1])

    confidence = (
        max(0.0, float(similarities[best])) * 70
        + max(
            0.0,
            float(similarities[best] - similarities[second])
        ) * 100
    )

    return {
        "intent": labels[best],
        "confidence": round(min(100.0, confidence), 1),
        "ranking": [
            {
                "intent": labels[int(i)],
                "score": round(
                    max(0.0, min(1.0, float(similarities[int(i)]))) * 100,
                    2,
                ),
            }
            for i in order[:3]
        ],
    }


def build_discovery_queries(query: str, intent: str) -> list[str]:
    additions = INTENT_EXPANSIONS.get(
        intent,
        INTENT_EXPANSIONS["Informação geral"],
    )
    return [
        query,
        f"{query} {additions[0]}",
        f"{query} {additions[1]}",
    ]

## 4. Metabusca e descoberta em múltiplas consultas

In [5]:
def web_search(
    query: str,
    max_results: int = 20,
    attempts: int = 3,
) -> list[WebResult]:

    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            raw = DDGS(timeout=10).text(
                query,
                region=REGION,
                safesearch="moderate",
                max_results=max_results,
                backend="auto",
            )

            results = []

            for rank, item in enumerate(raw, start=1):
                url = clean_text(item.get("href", ""))
                title = clean_text(item.get("title", ""))
                snippet = clean_text(item.get("body", ""))

                if url and title:
                    results.append(
                        WebResult(
                            rank=rank,
                            title=title,
                            url=url,
                            snippet=snippet,
                            domain=registered_domain(url),
                            discovery_query=query,
                        )
                    )

            if results:
                return results

        except Exception as exc:
            last_error = exc

        time.sleep(0.8 * attempt)

    if last_error:
        raise RuntimeError(f"Falha na busca web: {last_error}")

    return []


def multi_query_discovery(
    queries: list[str],
    results_per_query: int = 20,
) -> list[WebResult]:

    merged = []
    seen = set()

    for index, query in enumerate(queries, start=1):
        print(f"  descoberta {index}/{len(queries)}: {query}")

        try:
            results = web_search(
                query,
                max_results=results_per_query,
            )
        except Exception as exc:
            print("  aviso:", type(exc).__name__)
            continue

        for item in results:
            key = canonical_url(item.url)
            if key not in seen:
                seen.add(key)
                merged.append(item)

        time.sleep(0.25)

    return merged

## 5. Classificação automática do tipo de fonte

Tipos estimados:

- Oficial / Governo
- Acadêmica / Universidade
- Científica / Periódico
- Notícias / Editorial
- Comunidade / Fórum
- Comercial
- Independente / Blog
- Especializada / Nicho

O **sinal de nicho** não mede tráfego. Ele é apenas um indício usado para o algoritmo de diversidade.

In [6]:
TYPE_PATTERNS = {
    "Oficial / Governo": [
        r"\.gov\.", r"gov\.br$", r"governo",
        r"minist[eé]rio", r"secretaria", r"prefeitura",
    ],
    "Acadêmica / Universidade": [
        r"\.edu\.", r"\.edu$", r"\.ac\.",
        r"universidade", r"university", r"faculdade",
        r"instituto federal", r"reposit[oó]rio institucional",
    ],
    "Científica / Periódico": [
        r"doi\.org", r"scielo", r"pubmed", r"arxiv",
        r"journal", r"revista cient[ií]fica",
        r"periódico", r"proceedings", r"peer.?review",
    ],
    "Notícias / Editorial": [
        r"not[ií]cia", r"news", r"jornal",
        r"magazine", r"reportagem", r"editorial",
    ],
    "Comunidade / Fórum": [
        r"forum", r"f[oó]rum", r"reddit",
        r"stackexchange", r"stackoverflow",
        r"discuss", r"comunidade", r"pergunta",
    ],
    "Comercial": [
        r"comprar", r"pre[cç]o", r"produto",
        r"loja", r"store", r"shop",
        r"marketplace", r"frete", r"oferta",
    ],
    "Independente / Blog": [
        r"blog", r"wordpress", r"substack",
        r"medium\.com", r"blogspot",
        r"por [a-záàâãéêíóôõúç]",
    ],
}

NICHE_SIGNAL = {
    "Independente / Blog": 92,
    "Especializada / Nicho": 80,
    "Acadêmica / Universidade": 72,
    "Científica / Periódico": 68,
    "Oficial / Governo": 58,
    "Comunidade / Fórum": 55,
    "Notícias / Editorial": 42,
    "Comercial": 38,
}


def classify_source(
    domain: str,
    items: list[WebResult],
) -> SourceProfile:

    corpus = clean_text(
        " ".join(
            [domain]
            + [f"{x.title} {x.snippet}" for x in items[:8]]
        )
    ).casefold()

    scores = defaultdict(float)
    evidence = defaultdict(list)

    for source_type, patterns in TYPE_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, corpus, re.I):
                scores[source_type] += 1
                evidence[source_type].append(pattern)

    if domain.endswith((".gov.br", ".gov")):
        scores["Oficial / Governo"] += 4

    if domain.endswith((".edu", ".edu.br")):
        scores["Acadêmica / Universidade"] += 4

    if scores:
        source_type = max(scores, key=scores.get)
        confidence = (
            scores[source_type] / max(1.0, sum(scores.values()))
        ) * 100
    else:
        source_type = "Especializada / Nicho"
        confidence = 45.0

    niche = NICHE_SIGNAL[source_type]

    if source_type in {
        "Especializada / Nicho",
        "Independente / Blog",
    }:
        niche = min(100.0, niche + 5)

    return SourceProfile(
        source_type=source_type,
        confidence=round(confidence, 1),
        niche_signal=round(niche, 1),
        evidence=evidence[source_type][:5],
    )

## 6. Ranking contextual dos domínios

In [7]:
TYPE_CONTEXT = {
    "Informação geral": {
        "Especializada / Nicho": 1.06,
        "Oficial / Governo": 1.04,
        "Acadêmica / Universidade": 1.03,
        "Independente / Blog": 1.04,
    },
    "Pesquisa científica": {
        "Científica / Periódico": 1.10,
        "Acadêmica / Universidade": 1.08,
        "Oficial / Governo": 1.05,
    },
    "Notícias / atualidade": {
        "Notícias / Editorial": 1.10,
        "Oficial / Governo": 1.05,
        "Independente / Blog": 1.03,
        "Especializada / Nicho": 1.04,
    },
    "Prática / tutorial": {
        "Especializada / Nicho": 1.10,
        "Independente / Blog": 1.09,
        "Comunidade / Fórum": 1.05,
    },
    "Técnica / documentação": {
        "Especializada / Nicho": 1.09,
        "Acadêmica / Universidade": 1.05,
        "Oficial / Governo": 1.04,
        "Comunidade / Fórum": 1.03,
    },
    "Compra / comparação": {
        "Comercial": 1.08,
        "Notícias / Editorial": 1.05,
        "Comunidade / Fórum": 1.05,
        "Especializada / Nicho": 1.07,
        "Independente / Blog": 1.05,
    },
    "Opinião / comunidade": {
        "Comunidade / Fórum": 1.10,
        "Independente / Blog": 1.08,
        "Especializada / Nicho": 1.05,
    },
}


def build_site_candidates(
    original_query: str,
    discovery_queries: list[str],
    results: list[WebResult],
    intent: str,
) -> list[SiteCandidate]:

    scores = semantic_scores(
        original_query,
        [f"{x.title}. {x.snippet}" for x in results],
    )

    for item, score in zip(results, scores):
        item.semantic_score = score

    grouped = defaultdict(list)

    for item in results:
        if item.domain:
            grouped[item.domain].append(item)

    total = max(1, len(results))
    candidates = []

    for domain, items in grouped.items():
        items = sorted(items, key=lambda x: x.rank)
        profile = classify_source(domain, items)

        best_rank = min(x.rank for x in items)
        best_semantic = max(x.semantic_score for x in items)
        average_semantic = float(
            np.mean([x.semantic_score for x in items])
        )
        query_coverage = len({x.discovery_query for x in items})

        rank_score = min(
            100.0,
            (1 / math.log2(best_rank + 1.5)) * 100,
        )

        recurrence_score = min(
            100.0,
            (
                len(items)
                / max(3.0, total * 0.07)
            ) * 100,
        )

        coverage_score = (
            query_coverage / max(1, len(discovery_queries))
        ) * 100

        base_score = (
            0.44 * best_semantic
            + 0.26 * average_semantic
            + 0.16 * rank_score
            + 0.08 * recurrence_score
            + 0.06 * coverage_score
        )

        multiplier = TYPE_CONTEXT.get(
            intent, {}
        ).get(profile.source_type, 1.0)

        contextual_score = min(
            100.0,
            base_score * multiplier,
        )

        example = max(
            items,
            key=lambda x: (x.semantic_score, -x.rank),
        )

        descriptor = clean_text(
            " ".join(
                [domain, profile.source_type]
                + [
                    f"{x.title}. {x.snippet}"
                    for x in items[:4]
                ]
            )
        )[:5000]

        candidates.append(
            SiteCandidate(
                domain=domain,
                source_type=profile.source_type,
                type_confidence=profile.confidence,
                niche_signal=profile.niche_signal,
                base_score=round(base_score, 2),
                contextual_score=round(contextual_score, 2),
                selection_score=0.0,
                best_rank=best_rank,
                appearances=len(items),
                query_coverage=query_coverage,
                best_semantic_score=round(best_semantic, 2),
                average_semantic_score=round(average_semantic, 2),
                example_title=example.title,
                example_url=example.url,
                descriptor=descriptor,
            )
        )

    return sorted(
        candidates,
        key=lambda x: (
            x.contextual_score,
            x.best_semantic_score,
            -x.best_rank,
        ),
        reverse=True,
    )

## 7. Seleção diversificada dos cinco sites

O algoritmo usa um princípio semelhante a **Maximum Marginal Relevance**:

- relevância contextual é a base;
- repetir o mesmo tipo de fonte é penalizado;
- tipos novos recebem bônus;
- sinal de nicho recebe bônus moderado;
- sites semanticamente muito parecidos entre si recebem pequena penalização.

In [8]:
SEARCH_MODES = {
    "Relevância máxima": {
        "new_type_bonus": 2.0,
        "same_type_penalty": 1.5,
        "niche_weight": 0.010,
        "similarity_penalty": 2.0,
    },
    "Equilibrada": {
        "new_type_bonus": 5.0,
        "same_type_penalty": 3.5,
        "niche_weight": 0.035,
        "similarity_penalty": 4.0,
    },
    "Descoberta / web independente": {
        "new_type_bonus": 8.0,
        "same_type_penalty": 5.0,
        "niche_weight": 0.070,
        "similarity_penalty": 5.5,
    },
}


def select_diverse_sites(
    candidates: list[SiteCandidate],
    mode: str = "Equilibrada",
    top_n: int = 5,
) -> list[SiteCandidate]:

    if not candidates:
        return []

    settings = SEARCH_MODES[mode]
    pool = candidates[:30]
    embeddings = embed_texts([x.descriptor for x in pool])

    selected = []
    type_counts = defaultdict(int)

    while len(selected) < min(top_n, len(pool)):
        best_index = None
        best_utility = -1e9
        best_reason = ""

        for index, candidate in enumerate(pool):
            if index in selected:
                continue

            utility = candidate.contextual_score
            is_new_type = type_counts[candidate.source_type] == 0

            if is_new_type:
                utility += settings["new_type_bonus"]
            else:
                utility -= (
                    settings["same_type_penalty"]
                    * type_counts[candidate.source_type]
                )

            utility += (
                candidate.niche_signal
                * settings["niche_weight"]
            )

            if selected:
                max_similarity = max(
                    float(embeddings[index] @ embeddings[j])
                    for j in selected
                )
                utility -= (
                    max_similarity
                    * settings["similarity_penalty"]
                )

            reasons = [
                f"relevância contextual {candidate.contextual_score:.1f}",
                candidate.source_type,
            ]

            if is_new_type:
                reasons.append("aumenta diversidade")

            if candidate.niche_signal >= 75:
                reasons.append("forte sinal de nicho")

            if candidate.query_coverage > 1:
                reasons.append("apareceu em múltiplas buscas")

            if utility > best_utility:
                best_index = index
                best_utility = utility
                best_reason = "; ".join(reasons)

        if best_index is None:
            break

        chosen = pool[best_index]
        chosen.selection_score = round(best_utility, 2)
        chosen.why_selected = best_reason

        selected.append(best_index)
        type_counts[chosen.source_type] += 1

    return [pool[i] for i in selected]

## 8. Busca interna, robots.txt e extração

In [9]:
_ROBOTS_CACHE = {}


def robots_allowed(url: str) -> bool:
    try:
        parts = urlsplit(url)
        robots_url = (
            f"{parts.scheme}://{parts.netloc}/robots.txt"
        )

        if robots_url not in _ROBOTS_CACHE:
            parser = RobotFileParser()
            parser.set_url(robots_url)

            try:
                parser.read()
            except Exception:
                return True

            _ROBOTS_CACHE[robots_url] = parser

        return _ROBOTS_CACHE[robots_url].can_fetch(
            USER_AGENT,
            url,
        )

    except Exception:
        return True


def extract_page(url: str) -> tuple[str, str]:
    if not robots_allowed(url):
        return "", "bloqueado por robots.txt"

    try:
        response = SESSION.get(
            url,
            timeout=TIMEOUT,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            return "", f"HTTP {response.status_code}"

        content_type = response.headers.get(
            "content-type", ""
        ).lower()

        if "text/html" not in content_type:
            return "", "conteúdo não HTML"

        text = clean_text(
            trafilatura.extract(
                response.text,
                url=url,
                favor_precision=True,
                include_comments=False,
                include_tables=False,
            )
        )

        if not text:
            return "", "texto principal não extraído"

        if len(text) > MAX_EXTRACTED_CHARS:
            text = text[:MAX_EXTRACTED_CHARS].rstrip() + "…"

        return text, "ok"

    except requests.Timeout:
        return "", "timeout"

    except Exception as exc:
        return "", f"erro: {type(exc).__name__}"


def search_inside_site(
    domain: str,
    query: str,
    max_results: int,
) -> list[WebResult]:

    results = web_search(
        f"site:{domain} {query}",
        max_results=max_results,
    )

    return [
        x
        for x in results
        if x.domain == domain
        or x.domain.endswith("." + domain)
    ][:max_results]


def collect_final_results(
    query: str,
    sites: list[SiteCandidate],
    results_per_site: int,
    extract_content: bool = True,
) -> list[FinalResult]:

    collected = []
    lookup = {x.domain: x for x in sites}

    for position, site in enumerate(sites, start=1):
        print(f"  site {position}/{len(sites)}: {site.domain}")

        try:
            results = search_inside_site(
                site.domain,
                query,
                results_per_site,
            )
        except Exception as exc:
            print("    falha:", type(exc).__name__)
            continue

        for item in results:
            if extract_content:
                text, status = extract_page(item.url)
                time.sleep(0.25)
            else:
                text, status = "", "não solicitado"

            meta = lookup.get(item.domain, site)

            collected.append(
                FinalResult(
                    source_id=short_hash(canonical_url(item.url)),
                    site=item.domain,
                    source_type=meta.source_type,
                    title=item.title,
                    url=item.url,
                    snippet=item.snippet,
                    extracted_text=text,
                    extraction_status=status,
                    site_score=meta.selection_score,
                    semantic_score=0.0,
                    original_rank=item.rank,
                )
            )

    return collected

## 9. Ranking semântico e deduplicação

In [10]:
def rank_results(
    query: str,
    results: list[FinalResult],
) -> list[FinalResult]:

    if not results:
        return []

    documents = [
        clean_text(
            f"{x.title}. {x.snippet}. {x.extracted_text[:4000]}"
        )
        for x in results
    ]

    scores = semantic_scores(query, documents)

    for item, score in zip(results, scores):
        item.semantic_score = score

    return sorted(
        results,
        key=lambda x: (
            x.semantic_score,
            x.site_score,
            -x.original_rank,
        ),
        reverse=True,
    )


def normalize_title(title: str) -> str:
    normalized = clean_text(title).casefold()
    normalized = re.sub(r"[^\w\s]", " ", normalized)
    return re.sub(r"\s+", " ", normalized).strip()


def deduplicate_results(
    results: list[FinalResult],
    semantic_threshold: float = 0.94,
) -> list[FinalResult]:

    unique = []
    seen_urls = set()
    seen_titles = set()

    for item in results:
        url_key = canonical_url(item.url)
        title_key = normalize_title(item.title)

        if url_key in seen_urls:
            continue

        if title_key and title_key in seen_titles:
            continue

        seen_urls.add(url_key)
        seen_titles.add(title_key)
        unique.append(item)

    if len(unique) <= 1:
        return unique

    embeddings = embed_texts(
        [f"{x.title}. {x.snippet}" for x in unique]
    )

    keep = []

    for index in range(len(unique)):
        duplicate = any(
            float(embeddings[index] @ embeddings[j])
            >= semantic_threshold
            for j in keep
        )

        if not duplicate:
            keep.append(index)

    return [unique[i] for i in keep]

## 10. Síntese extrativa e referências

In [11]:
def split_sentences(text: str) -> list[str]:
    parts = re.split(
        r"(?<=[.!?])\s+(?=[A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ0-9])",
        clean_text(text),
    )

    return [
        clean_text(part)
        for part in parts
        if 45 <= len(clean_text(part)) <= 500
    ]


def create_synthesis(
    query: str,
    results: list[FinalResult],
    max_sentences: int = 7,
    diversity_threshold: float = 0.87,
):

    candidates = []

    for item in results[:15]:
        source_text = item.extracted_text or item.snippet

        for sentence in split_sentences(source_text)[:22]:
            candidates.append({
                "sentence": sentence,
                "source_id": item.source_id,
                "site": item.site,
                "source_type": item.source_type,
                "title": item.title,
                "url": item.url,
            })

    if not candidates:
        return [], pd.DataFrame()

    sentences = [x["sentence"] for x in candidates]
    embeddings = embed_texts(sentences)
    query_embedding = embed_texts([query])[0]
    relevance = embeddings @ query_embedding

    order = np.argsort(relevance)[::-1]
    selected = []
    domain_counts = defaultdict(int)

    for value in order:
        index = int(value)

        if len(selected) >= max_sentences:
            break

        if domain_counts[candidates[index]["site"]] >= 2:
            continue

        redundant = any(
            float(embeddings[index] @ embeddings[j])
            >= diversity_threshold
            for j in selected
        )

        if redundant:
            continue

        selected.append(index)
        domain_counts[candidates[index]["site"]] += 1

    synthesis = []

    for index in selected:
        item = candidates[index].copy()
        item["relevance"] = round(
            max(0.0, min(1.0, float(relevance[index]))) * 100,
            2,
        )
        synthesis.append(item)

    used_ids = {x["source_id"] for x in synthesis}

    sources = [
        {
            "fonte": item.source_id,
            "tipo": item.source_type,
            "site": item.site,
            "título": item.title,
            "relevância": item.semantic_score,
            "url": item.url,
        }
        for item in results
        if item.source_id in used_ids
    ]

    sources_df = (
        pd.DataFrame(sources).drop_duplicates("fonte")
        if sources
        else pd.DataFrame()
    )

    return synthesis, sources_df

## 11. Pipeline completo

In [12]:
def adaptive_context_search(
    query: str,
    mode: str = "Equilibrada",
    results_per_site: int = 5,
    discovery_depth: int = 20,
    extract_content: bool = True,
    summary_sentences: int = 7,
) -> dict:

    print("1/8 — Interpretando contexto...")
    intent = infer_search_intent(query)
    print("  ", intent["intent"], f"({intent['confidence']}%)")

    queries = build_discovery_queries(
        query,
        intent["intent"],
    )

    print("2/8 — Descoberta web...")
    discovery = multi_query_discovery(
        queries,
        discovery_depth,
    )

    if not discovery:
        raise RuntimeError(
            "A descoberta não retornou resultados."
        )

    print("3/8 — Classificando fontes...")
    candidates = build_site_candidates(
        query,
        queries,
        discovery,
        intent["intent"],
    )

    print("4/8 — Selecionando 5 fontes diversas...")
    sites = select_diverse_sites(
        candidates,
        mode,
        TOP_SITES,
    )

    if not sites:
        raise RuntimeError(
            "Não foi possível selecionar sites."
        )

    for index, site in enumerate(sites, start=1):
        print(
            f"  {index}. {site.domain} — {site.source_type}"
        )

    print("5/8 — Buscando dentro dos sites...")
    raw_results = collect_final_results(
        query,
        sites,
        results_per_site,
        extract_content,
    )

    print("6/8 — Ranking semântico...")
    ranked = rank_results(
        query,
        raw_results,
    )

    print("7/8 — Deduplicação...")
    unique = deduplicate_results(ranked)

    print("8/8 — Síntese referenciada...")
    synthesis, sources = create_synthesis(
        query,
        unique,
        summary_sentences,
    )

    return {
        "query": query,
        "mode": mode,
        "intent": intent,
        "discovery_queries": queries,
        "discovery_results": discovery,
        "all_candidates": candidates,
        "sites": sites,
        "results": unique,
        "synthesis": synthesis,
        "synthesis_sources": sources,
    }

## 12. Visualização em tabela e cards

In [13]:
def sites_dataframe(sites):
    return pd.DataFrame([
        {
            "posição": index,
            "site": item.domain,
            "tipo": item.source_type,
            "confiança_tipo": item.type_confidence,
            "sinal_nicho": item.niche_signal,
            "score_contextual": item.contextual_score,
            "score_seleção": item.selection_score,
            "aparições": item.appearances,
            "cobertura_consultas": item.query_coverage,
            "por_que_foi_escolhido": item.why_selected,
            "página_representativa": item.example_title,
            "link_representativo": item.example_url,
        }
        for index, item in enumerate(sites, start=1)
    ])


def results_dataframe(results):
    return pd.DataFrame([
        {
            "fonte": item.source_id,
            "site": item.site,
            "tipo": item.source_type,
            "relevância": item.semantic_score,
            "título": item.title,
            "resumo": item.snippet,
            "status_extração": item.extraction_status,
            "texto_extraído": item.extracted_text,
            "url": item.url,
        }
        for item in results
    ])


def relevance_label(score: float) -> str:
    if score >= 75:
        return "Muito alta"
    if score >= 60:
        return "Alta"
    if score >= 45:
        return "Média"
    if score >= 30:
        return "Baixa"
    return "Muito baixa"


def result_information(
    item: FinalResult,
    max_chars: int = 900,
) -> tuple[str, str]:
    if item.extracted_text:
        text = clean_text(item.extracted_text)
        origin = "Conteúdo extraído da página"
    elif item.snippet:
        text = clean_text(item.snippet)
        origin = "Resumo retornado pelo mecanismo de busca"
    else:
        text = "Nenhuma descrição textual foi obtida para esta página."
        origin = "Sem descrição disponível"

    if len(text) > max_chars:
        text = text[:max_chars].rsplit(" ", 1)[0].rstrip() + "…"

    return text, origin


def display_selected_source_cards(
    sites: list[SiteCandidate],
):
    """
    Mostra as 5 fontes selecionadas com a página específica
    que justificou a escolha do domínio.
    """
    if not sites:
        display(Markdown("Nenhuma fonte foi selecionada."))
        return

    cards = []

    for index, site in enumerate(sites, start=1):
        domain = html.escape(site.domain)
        source_type = html.escape(site.source_type)
        title = html.escape(site.example_title or "Página representativa")
        url = html.escape(site.example_url or "", quote=True)
        reason = html.escape(site.why_selected or "")
        open_link = (
            f'<a href="{url}" target="_blank" rel="noopener noreferrer">'
            'Abrir página sugerida</a>'
            if url
            else "Link não disponível"
        )

        cards.append(
            f"""
            <div style="
                border:1px solid #d9d9d9;
                border-radius:14px;
                padding:16px 18px;
                margin:12px 0;
                font-family:Arial,sans-serif;
                line-height:1.45;
            ">
                <div style="
                    font-size:12px;
                    opacity:.72;
                    margin-bottom:5px;
                ">
                    Fonte #{index} · {source_type} · {domain}
                </div>

                <div style="
                    font-size:18px;
                    font-weight:650;
                    margin-bottom:7px;
                ">
                    <a
                        href="{url}"
                        target="_blank"
                        rel="noopener noreferrer"
                    >
                        {title}
                    </a>
                </div>

                <div style="
                    font-size:13px;
                    margin-bottom:8px;
                ">
                    <strong>Score contextual:</strong>
                    {site.contextual_score:.1f}
                    &nbsp;·&nbsp;
                    <strong>Score de seleção:</strong>
                    {site.selection_score:.1f}
                </div>

                <div style="
                    font-size:13px;
                    margin-bottom:8px;
                ">
                    <strong>Por que esta fonte foi escolhida:</strong>
                    {reason}
                </div>

                <div style="
                    font-size:13px;
                    margin-bottom:4px;
                ">
                    {open_link}
                </div>

                <div style="
                    font-size:11px;
                    opacity:.65;
                    word-break:break-all;
                ">
                    {url}
                </div>
            </div>
            """
        )

    display(HTML("".join(cards)))


def display_result_cards(
    results: list[FinalResult],
    limit: int = 15,
):
    """
    Mostra páginas específicas encontradas dentro das 5 fontes.
    Cada resultado tem link direto para a página sugerida.
    """
    if not results:
        display(Markdown("Nenhum resultado foi encontrado."))
        return

    cards = []

    for index, item in enumerate(results[:limit], start=1):
        info_text, info_origin = result_information(item)

        title = html.escape(item.title)
        site = html.escape(item.site)
        source_type = html.escape(item.source_type)
        url = html.escape(item.url, quote=True)
        information = html.escape(info_text)
        information_origin = html.escape(info_origin)
        extraction_status = html.escape(item.extraction_status)

        relevance = float(item.semantic_score)
        relevance_text = relevance_label(relevance)

        cards.append(
            f"""
            <div style="
                border:1px solid #d9d9d9;
                border-radius:14px;
                padding:18px 20px;
                margin:14px 0;
                font-family:Arial,sans-serif;
                line-height:1.5;
            ">
                <div style="
                    font-size:12px;
                    opacity:.72;
                    margin-bottom:6px;
                ">
                    Resultado #{index}
                    &nbsp;·&nbsp;
                    {source_type}
                    &nbsp;·&nbsp;
                    {site}
                </div>

                <div style="
                    font-size:19px;
                    font-weight:650;
                    margin-bottom:8px;
                ">
                    <a
                        href="{url}"
                        target="_blank"
                        rel="noopener noreferrer"
                    >
                        {title}
                    </a>
                </div>

                <div style="
                    display:flex;
                    gap:18px;
                    flex-wrap:wrap;
                    font-size:13px;
                    margin-bottom:12px;
                ">
                    <div>
                        <strong>Relevância para o tema:</strong>
                        {relevance:.1f}% ({relevance_text})
                    </div>

                    <div>
                        <strong>Extração:</strong>
                        {extraction_status}
                    </div>
                </div>

                <div style="
                    font-size:14px;
                    margin-bottom:6px;
                    font-weight:600;
                ">
                    Informações encontradas
                </div>

                <div style="
                    font-size:13px;
                    line-height:1.55;
                    margin-bottom:9px;
                ">
                    {information}
                </div>

                <div style="
                    font-size:11px;
                    opacity:.65;
                    margin-bottom:10px;
                ">
                    {information_origin}
                </div>

                <div style="
                    font-size:13px;
                    margin-bottom:4px;
                ">
                    <a
                        href="{url}"
                        target="_blank"
                        rel="noopener noreferrer"
                    >
                        Abrir página original
                    </a>
                </div>

                <div style="
                    font-size:11px;
                    opacity:.65;
                    word-break:break-all;
                ">
                    {url}
                </div>
            </div>
            """
        )

    display(HTML("".join(cards)))


def display_synthesis(synthesis, sources):
    display(Markdown("## Síntese rastreável"))

    if not synthesis:
        display(
            Markdown(
                "Não houve texto suficiente para gerar síntese."
            )
        )
        return

    lines = [
        f"- {x['sentence']} **[{x['source_id']}]**"
        for x in synthesis
    ]

    display(Markdown("\n".join(lines)))

    if not sources.empty:
        display(Markdown("### Fontes usadas"))
        display(sources)

## 13. Interface do usuário

In [14]:
query_input = widgets.Text(
    value="",
    placeholder="Digite qualquer assunto...",
    description="Assunto:",
    layout=widgets.Layout(width="88%"),
    style={"description_width": "80px"},
)

mode_input = widgets.Dropdown(
    options=list(SEARCH_MODES),
    value="Equilibrada",
    description="Modo:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="55%"),
)

results_input = widgets.IntSlider(
    value=5,
    min=2,
    max=10,
    step=1,
    description="Por site:",
    continuous_update=False,
    style={"description_width": "80px"},
)

depth_input = widgets.IntSlider(
    value=20,
    min=10,
    max=30,
    step=5,
    description="Descoberta:",
    continuous_update=False,
    style={"description_width": "80px"},
)

extract_input = widgets.Checkbox(
    value=True,
    description="Extrair conteúdo das páginas",
)

summary_input = widgets.IntSlider(
    value=7,
    min=3,
    max=12,
    step=1,
    description="Síntese:",
    continuous_update=False,
    style={"description_width": "80px"},
)

search_button = widgets.Button(
    description="Buscar",
    button_style="primary",
    icon="search",
)

output = widgets.Output()

LAST_RUN = None
LAST_SITES_DF = pd.DataFrame()
LAST_RESULTS_DF = pd.DataFrame()


def handle_search(_):
    global LAST_RUN
    global LAST_SITES_DF
    global LAST_RESULTS_DF

    query = query_input.value.strip()

    with output:
        clear_output()

        if not query:
            print("Digite um assunto.")
            return

        try:
            run = adaptive_context_search(
                query=query,
                mode=mode_input.value,
                results_per_site=results_input.value,
                discovery_depth=depth_input.value,
                extract_content=extract_input.value,
                summary_sentences=summary_input.value,
            )

            LAST_RUN = run
            LAST_SITES_DF = sites_dataframe(run["sites"])
            LAST_RESULTS_DF = results_dataframe(run["results"])

            clear_output()

            display(Markdown("# Resultado"))

            display(Markdown(
                f"**Busca:** {query}  \n"
                f"**Contexto inferido:** {run['intent']['intent']}  \n"
                f"**Confiança:** {run['intent']['confidence']}%  \n"
                f"**Modo:** {run['mode']}"
            ))

            display(
                Markdown(
                    "## 5 fontes selecionadas "
                    "e páginas que justificaram a escolha"
                )
            )

            display_selected_source_cards(
                run["sites"]
            )

            display(
                Markdown(
                    "## Páginas sugeridas pelo buscador"
                )
            )

            display_result_cards(
                run["results"],
                limit=15,
            )

            display_synthesis(
                run["synthesis"],
                run["synthesis_sources"],
            )

            display(
                Markdown(
                    "### Tabela completa dos resultados"
                )
            )

            display(LAST_RESULTS_DF)

        except Exception as exc:
            clear_output()
            print("Não foi possível concluir a busca:")
            print(type(exc).__name__, ":", exc)


search_button.on_click(handle_search)

display(
    widgets.VBox([
        query_input,
        mode_input,
        results_input,
        depth_input,
        extract_input,
        summary_input,
        search_button,
        output,
    ])
)

## 14. Diagnóstico e exportação

In [17]:
def export_last_search():
    if LAST_RUN is None:
        print("Faça uma busca primeiro.")
        return

    sites_csv = "/content/sites_selecionados_v5.csv"
    results_csv = "/content/resultados_v5.csv"
    excel_path = "/content/buscador_adaptativo_v5.xlsx"
    synthesis_path = "/content/sintese_referenciada_v5.txt"

    LAST_SITES_DF.to_csv(
        sites_csv,
        index=False,
        encoding="utf-8-sig",
    )

    LAST_RESULTS_DF.to_csv(
        results_csv,
        index=False,
        encoding="utf-8-sig",
    )

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl",
    ) as writer:
        LAST_SITES_DF.to_excel(
            writer,
            sheet_name="5 fontes",
            index=False,
        )
        LAST_RESULTS_DF.to_excel(
            writer,
            sheet_name="resultados",
            index=False,
        )

    lines = [
        f"BUSCA: {LAST_RUN['query']}",
        f"CONTEXTO: {LAST_RUN['intent']['intent']}",
        f"MODO: {LAST_RUN['mode']}",
        "",
        "SÍNTESE",
        "",
    ]

    lines.extend([
        f"- {x['sentence']} [{x['source_id']}]"
        for x in LAST_RUN["synthesis"]
    ])

    lines.extend(["", "FONTES", ""])

    sources = LAST_RUN["synthesis_sources"]

    if not sources.empty:
        lines.extend([
            (
                f"[{row['fonte']}] {row['tipo']} — "
                f"{row['título']} — {row['url']}"
            )
            for _, row in sources.iterrows()
        ])

    with open(
        synthesis_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write("\n".join(lines))

    print(sites_csv)
    print(results_csv)
    print(excel_path)
    print(synthesis_path)


# Execute depois de uma busca:
export_last_search()

# Diagnóstico opcional:
display(pd.DataFrame(LAST_RUN["intent"]["ranking"]))
display(pd.DataFrame([
    {
        "site": x.domain,
        "tipo": x.source_type,
        "nicho": x.niche_signal,
        "score_base": x.base_score,
        "score_contextual": x.contextual_score,
    }
    for x in LAST_RUN["all_candidates"][:20]
]))

/content/sites_selecionados_v5.csv
/content/resultados_v5.csv
/content/buscador_adaptativo_v5.xlsx
/content/sintese_referenciada_v5.txt


,intent,score
0,Opinião / comunidade,41.37
1,Pesquisa científica,40.97
2,Prática / tutorial,38.56


,site,tipo,nicho,score_base,score_contextual
0,congesp.rn.gov.br,Oficial / Governo,58,68.26,68.26
1,cecierj.edu.br,Acadêmica / Universidade,72,64.40,64.40
2,udsp.org.br,Especializada / Nicho,85,61.02,64.07
3,univap.br,Independente / Blog,97,55.94,60.42
4,editorarealize.com.br,Especializada / Nicho,85,56.75,59.59
5,even3.com.br,Especializada / Nicho,85,55.93,58.73
6,ufg.br,Especializada / Nicho,85,55.66,58.44
7,scielo.br,Científica / Periódico,68,58.05,58.05
8,educacaobasicarevista.com.br,Independente / Blog,97,53.27,57.53
9,rubeus.com.br,Independente / Blog,97,53.07,57.31


## 15. Princípio final da v5

O buscador tenta responder:

> **Quais cinco fontes encontradas formam o conjunto mais relevante, útil e complementar para esta necessidade?**

```text
RELEVÂNCIA SEMÂNTICA
       +
CONTEXTO
       +
TIPO DE FONTE
       +
DIVERSIDADE
       +
SINAL DE NICHO
```

Um blog não entra simplesmente por ser pequeno. Ele entra quando é relevante **e** acrescenta diversidade.

### Limitações assumidas
- mecanismos gratuitos podem aplicar rate limit;
- alguns sites bloqueiam automação;
- páginas muito dependentes de JavaScript podem não ser extraídas;
- login/paywall não é contornado;
- `robots.txt` pode impedir coleta;
- tipo de fonte e sinal de nicho são estimativas;
- a descoberta depende dos resultados retornados pela metabusca;
- a síntese é extrativa e prioriza rastreabilidade.

In [18]:
DEMO_MAX_RESULTS = 10


def _github_demo_excerpt(
    item: FinalResult,
    max_chars: int = 700,
) -> tuple[str, str]:
    if item.extracted_text:
        text = clean_text(item.extracted_text)
        origin = "Conteúdo extraído da página"
    elif item.snippet:
        text = clean_text(item.snippet)
        origin = "Resumo retornado pelo mecanismo de busca"
    else:
        text = "Nenhuma descrição textual foi obtida para esta página."
        origin = "Sem descrição disponível"

    if len(text) > max_chars:
        text = text[:max_chars]
        if " " in text:
            text = text.rsplit(" ", 1)[0]
        text = text.rstrip() + "…"

    return text, origin


def display_static_github_demo(
    run: Optional[dict] = None,
    max_results: int = DEMO_MAX_RESULTS,
):
    if run is None:
        run = LAST_RUN

    if run is None:
        display(
            Markdown(
                "### Demonstração estática\n"
                "Faça uma busca primeiro e depois execute esta célula."
            )
        )
        return

    query = run.get("query", "")
    mode = run.get("mode", "")
    intent_data = run.get("intent", {}) or {}
    sites = run.get("sites", []) or []
    results = run.get("results", []) or []
    synthesis = run.get("synthesis", []) or []
    sources = run.get("synthesis_sources", pd.DataFrame())

    display(
        Markdown(
            "# Demonstração de execução\n\n"
            f"**Busca realizada:** {query}  \n"
            f"**Contexto inferido:** {intent_data.get('intent', '')}  \n"
            f"**Confiança da interpretação:** "
            f"{intent_data.get('confidence', '')}%  \n"
            f"**Modo de busca:** {mode}"
        )
    )

    display(
        Markdown(
            "## 1. Cinco fontes selecionadas dinamicamente"
        )
    )

    for index, site in enumerate(sites, start=1):
        page_title = clean_text(
            site.example_title or "Página representativa"
        )
        page_url = clean_text(site.example_url or "")
        reason = clean_text(site.why_selected or "")

        if page_url:
            page_line = (
                f"**Página representativa:** "
                f"[{page_title}]({page_url})"
            )
        else:
            page_line = (
                f"**Página representativa:** {page_title}"
            )

        display(
            Markdown(
                f"### Fonte {index} — {site.domain}\n\n"
                f"**Tipo de fonte:** {site.source_type}  \n"
                f"**Score contextual:** {site.contextual_score:.2f}  \n"
                f"**Score de seleção:** {site.selection_score:.2f}  \n"
                f"{page_line}  \n"
                f"**Link direto:** {page_url or 'não disponível'}  \n"
                f"**Por que foi escolhida:** {reason}"
            )
        )

    selected_results = results[:max_results]

    display(
        Markdown(
            "## 2. Páginas sugeridas pelo buscador\n\n"
            f"**Resultados exibidos:** {len(selected_results)}"
        )
    )

    for index, item in enumerate(
        selected_results,
        start=1,
    ):
        excerpt, origin = _github_demo_excerpt(item)
        relevance = float(item.semantic_score)

        if relevance >= 75:
            label = "Muito alta"
        elif relevance >= 60:
            label = "Alta"
        elif relevance >= 45:
            label = "Média"
        elif relevance >= 30:
            label = "Baixa"
        else:
            label = "Muito baixa"

        display(
            Markdown(
                f"### {index}. [{item.title}]({item.url})\n\n"
                f"**Fonte:** {item.site}  \n"
                f"**Tipo:** {item.source_type}  \n"
                f"**Relevância para o tema:** "
                f"{relevance:.1f}% ({label})  \n"
                f"**Status da extração:** "
                f"{item.extraction_status}  \n\n"
                f"**Informações encontradas:**  \n"
                f"{excerpt}  \n\n"
                f"*{origin}*  \n\n"
                f"**Abrir página original:** {item.url}"
            )
        )
        display(Markdown("---"))

    display(
        Markdown(
            "## 3. Síntese rastreável"
        )
    )

    if synthesis:
        lines = [
            f"- {item['sentence']} **[{item['source_id']}]**"
            for item in synthesis
        ]
        display(Markdown("\n".join(lines)))
    else:
        display(
            Markdown(
                "Não houve conteúdo suficiente para gerar a síntese."
            )
        )

    display(
        Markdown(
            "## 4. Referências utilizadas na síntese"
        )
    )

    if (
        isinstance(sources, pd.DataFrame)
        and not sources.empty
    ):
        for _, row in sources.iterrows():
            source_id = row.get("fonte", "")
            title = row.get("título", "")
            url = row.get("url", "")
            source_type = row.get("tipo", "")
            site = row.get("site", "")
            relevance = row.get("relevância", "")

            display(
                Markdown(
                    f"**[{source_id}] [{title}]({url})**  \n"
                    f"Tipo: {source_type}  \n"
                    f"Site: {site}  \n"
                    f"Relevância: {relevance}  \n"
                    f"URL: {url}"
                )
            )
            display(Markdown("---"))
    else:
        display(
            Markdown(
                "Nenhuma referência adicional foi registrada."
            )
        )

    display(
        Markdown(
            "## 5. Observação\n\n"
            "Esta demonstração foi gerada a partir de uma execução real "
            "do notebook. Os links acima apontam para as páginas "
            "específicas encontradas pelo buscador."
        )
    )


display_static_github_demo()

# Demonstração de execução

**Busca realizada:** uso de metodologias ativas em instituições públicas  
**Contexto inferido:** Opinião / comunidade  
**Confiança da interpretação:** 29.4%  
**Modo de busca:** Equilibrada

## 1. Cinco fontes selecionadas dinamicamente

### Fonte 1 — congesp.rn.gov.br

**Tipo de fonte:** Oficial / Governo  
**Score contextual:** 68.26  
**Score de seleção:** 75.29  
**Página representativa:** [PDF Metodologias Ativas E Inovação De Processos Na Gestão Pública ...](https://congesp.rn.gov.br/anais/v-17/gestao-publica-tecnologia-e-inovacao/metodologias-ativas-e-inovacao-de-processos-na-gestao-publica-experiencias-na-formacao-continuada-na-escola-de-governo-do-rn.pdf)  
**Link direto:** https://congesp.rn.gov.br/anais/v-17/gestao-publica-tecnologia-e-inovacao/metodologias-ativas-e-inovacao-de-processos-na-gestao-publica-experiencias-na-formacao-continuada-na-escola-de-governo-do-rn.pdf  
**Por que foi escolhida:** relevância contextual 68.3; Oficial / Governo; aumenta diversidade

### Fonte 2 — udsp.org.br

**Tipo de fonte:** Especializada / Nicho  
**Score contextual:** 64.07  
**Score de seleção:** 69.39  
**Página representativa:** [Metodologias Ativas: São aplicáveis na educação pública? – UDSP](https://udsp.org.br/2025/08/25/metodologias-ativas/)  
**Link direto:** https://udsp.org.br/2025/08/25/metodologias-ativas/  
**Por que foi escolhida:** relevância contextual 64.1; Especializada / Nicho; aumenta diversidade; forte sinal de nicho

### Fonte 3 — cecierj.edu.br

**Tipo de fonte:** Acadêmica / Universidade  
**Score contextual:** 64.40  
**Score de seleção:** 68.78  
**Página representativa:** [Revista Educação Pública - O uso das metodologias ativas de aprendizagem na formação do professor: das universidades para a prática nas escolas](https://educacaopublica.cecierj.edu.br/artigos/23/8/o-uso-das-metodologias-ativas-de-aprendizagem-na-formacao-do-professor-das-universidades-para-a-pratica-nas-escolas)  
**Link direto:** https://educacaopublica.cecierj.edu.br/artigos/23/8/o-uso-das-metodologias-ativas-de-aprendizagem-na-formacao-do-professor-das-universidades-para-a-pratica-nas-escolas  
**Por que foi escolhida:** relevância contextual 64.4; Acadêmica / Universidade; aumenta diversidade; apareceu em múltiplas buscas

...

**[50b286d5] [Inovação no ensino: uma revisão sistemática das metodologias ...](https://www.scielo.br/j/aval/a/C9khps4n4BnGj6ZWkZvBk9z/?format=html)**  
Tipo: Científica / Periódico  
Site: scielo.br  
Relevância: 58.08  
URL: https://www.scielo.br/j/aval/a/C9khps4n4BnGj6ZWkZvBk9z/?format=html

---

**[5d6d5a7f] [Metodologias Ativas: bases, contrapontos e (con ... - Brasil](https://www.scielo.br/j/edreal/a/Ffpm3CCKJSZmZ4btLMJzXvk/?format=html)**  
Tipo: Científica / Periódico  
Site: scielo.br  
Relevância: 57.82  
URL: https://www.scielo.br/j/edreal/a/Ffpm3CCKJSZmZ4btLMJzXvk/?format=html

---

## 5. Observação

Esta demonstração foi gerada a partir de uma execução real do notebook. Os links acima apontam para as páginas específicas encontradas pelo buscador.